# FBI-CA — Simulation Demo
**Factor-Based Imputation via Cross-sectional Averages**
For bootstrap intervals see **`03_bootstrap_validation.ipynb`**. Enjoy!

In [ ]:
# Install the package (run once)
# !pip install -e ..     # from the notebooks/ folder
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from fbica import FBICA, generate_panel, generate_missing, run_simulation
from fbica.dgp import draw_fixed_factors_and_loadings
from fbica.metrics import rmse, mae, summary_table

np.random.seed(0)

## 1. DGP parameters


In [ ]:
# DGP parameters
N   = 50      # cross-sectional units
T   = 100     # time periods
m   = 4       # variables 
r   = 3       # true latent factors
phi = 0.3     # AR(1) persistence
pi  = 0.0     # spatial dependence 
miss_probs = [0.10, 0.13, 0.16, 0.20]

print(f"Panel: T={T}  N={N}  m={m}  r={r}")
print(f"phi={phi}  pi={pi}")
print(f"Missingness probabilities: {miss_probs}")


## 2. Generate one sample dataset

In [ ]:

X, F_true, lam_true, nu = generate_panel(
    N=N, T=T, m=m, r=r, phi=phi, pi=pi, seed_errors=42
)
X_obs, miss = generate_missing(X, miss_probs=miss_probs, seed=42)
frac_missing = miss.mean()
print(f"Fraction missing: {frac_missing}")


## 3. Impute with FBI-CA

In [ ]:

imp_loo = FBICA(use_loo=True)
X_loo   = imp_loo.fit_transform(X_obs)
imp_plain = FBICA(use_loo=False)
X_plain   = imp_plain.fit_transform(X_obs)
print("RMSE at ALL missing entries:")
print(f"  LOO   : {rmse(X, X_loo,   miss)}")
print(f"  Plain : {rmse(X, X_plain, miss)}")


In [ ]:
print(summary_table(X, X_loo, miss,
                    var_names=[f"b={b}" for b in range(m)]))


## 4. Plot: true vs imputed at fixed cells

In [ ]:

state_idx = 5
var_idx   = 0

obs_times  = ~miss[:, state_idx, var_idx]
miss_times = miss[:, state_idx, var_idx]

fig, ax = plt.subplots(figsize=(12, 4))
t_axis = np.arange(T)

ax.plot(t_axis, X[:, state_idx, var_idx],  'k--', lw=1, label='True x', alpha=0.5)
ax.scatter(t_axis[obs_times],  X_obs[obs_times,  state_idx, var_idx], s=15, c='steelblue', label='Observed')
ax.scatter(t_axis[miss_times], X_loo[miss_times, state_idx, var_idx], s=30, c='red',       label='Imputed (LOO)', zorder=3)

ax.set_xlabel("Time period")
ax.set_ylabel("Value")
ax.set_title(f"Unit {state_idx}, Variable {var_idx}")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Monte Carlo

In [ ]:
grid  = [(25, 25), (50, 50), (100, 100)]
n_sim = 200

rows = []
for (Ng, Tg) in grid:
    res = run_simulation(
        N=Ng, T=Tg, m=m, r=r, phi=phi, pi=pi,
        miss_probs=miss_probs, n_sim=n_sim,
        use_loo=True, verbose=False,
    )
    rows.append({
        "N": Ng, "T": Tg,
        "True C":      round(float(res["true_C"][0]), 3),
        "Mean imputed": round(float(res["mean_imputed"][0]), 3),
        "RMSE":         round(float(res["rmse"][0]), 4),
        "Bias":         round(float(res["bias"][0]), 4),
    })
    print(f"N={Ng}  T={Tg}  RMSE={rows[-1]['RMSE']}")

pd.DataFrame(rows)


## 6. Compare LOO vs plain

`compare_loo_vs_plain` runs the same experiment twice.


In [ ]:
from fbica.simulation import compare_loo_vs_plain

comp = compare_loo_vs_plain(
    N=50, T=50, m=m, r=r, phi=phi, pi=pi,
    miss_probs=miss_probs, n_sim=200, verbose=False,
)

print(f"LOO   RMSE: {comp['loo']['rmse'][0]:.4f}")
print(f"Plain RMSE: {comp['plain']['rmse'][0]:.4f}")


## 7. Spatial dependence (pi > 0)

Repeat with `pi=0.2` to introduce cross-sectional error correlation.


In [ ]:
res_nocs = run_simulation(N=50, T=50, m=m, r=r,
                           phi=phi, pi=0.0, miss_probs=miss_probs,
                           n_sim=200, use_loo=True, verbose=False)
res_cs   = run_simulation(N=50, T=50, m=m, r=r,
                           phi=phi, pi=0.2, miss_probs=miss_probs,
                           n_sim=200, use_loo=True, verbose=False)

print(f"pi=0.0  RMSE: {res_nocs['rmse'][0]}")
print(f"pi=0.2  RMSE: {res_cs['rmse'][0]}")


## 8. Loading-dependent missingness

Pass a custom `forced_missing` and evaluate RMSE at specific cells of interest.


In [ ]:
fixed = [(4, 4, 0), (10, 15, 1), (20, 20, 2)]

res = run_simulation(
    N=50, T=50, m=m, r=r, phi=phi, pi=pi,
    miss_probs=miss_probs, n_sim=200, use_loo=True,
    fixed_points=fixed, verbose=False,
)

for j, (t, i, b) in enumerate(fixed):
    print(f"  Cell (t={t}, i={i}, b={b})"
          f"  true_C={res['true_C'][j]}"
          f"  RMSE={res['rmse'][j]}"
          f"  bias={res['bias'][j]}")
